## A1

In [1]:
import random
import math

def buffon_pi(n, L=1.0,D=2.0):
    hits = 0

    for i in range(n):
        x = (D/2.0) * random.random()
        theta = (math.pi/2.0) * random.random()

        condition = (L/2.0) * math.sin(theta)

        if x <= condition:
            hits+=1

    # print(hits)
    # print(n)

    p_hat = hits/n
    pi_est = (2.0 * L) / (D * p_hat)

    # print(p_hat)
    # print(pi_est)

    abs_error = abs(math.pi - pi_est)
    
    
    return hits, p_hat, pi_est, abs_error

N = [100,1000,10000,100000,1000000]

for n in N:
    print(f"=" * 60)
    print(f"N Drops: {n}")
    hits, p_hat, pi_est, abs_error = buffon_pi(n)
    print(f"Hits: {hits}")
    print(f"P_hat: {p_hat}")
    print(f"pi_est: {pi_est}")
    print(f"Abs error: {abs_error}")




N Drops: 100
Hits: 33
P_hat: 0.33
pi_est: 3.0303030303030303
Abs error: 0.11128962328676284
N Drops: 1000
Hits: 325
P_hat: 0.325
pi_est: 3.0769230769230766
Abs error: 0.06466957666671647
N Drops: 10000
Hits: 3249
P_hat: 0.3249
pi_est: 3.077870113881194
Abs error: 0.06372253970859898
N Drops: 100000
Hits: 31763
P_hat: 0.31763
pi_est: 3.148317224443535
Abs error: 0.006724570853741785
N Drops: 1000000
Hits: 318534
P_hat: 0.318534
pi_est: 3.1393822951396086
Abs error: 0.0022103584501844686


## B2

In [2]:
import math

def msws_generator_step(x,w,c,m):
    w = (w+c) % m
    x = (x*x + w) % m

    u = (x//(1<<16))/(1<<16)
    return x,w,u

def msws_generator(n_samples,x0=0,w0=0,c=0xb5ad4ec5,m=2**32):
    x = x0
    w = w0
    U = []

    for _ in range(n_samples):
        x,w,u = msws_generator_step(x,w,c,m)
        U.append(u)
    return U

# print(msws_generator(10))


N = 10000
U = msws_generator(4*N)

total = 0

for k in range(N):
    u1 = U[4*k]
    u2 = U[4*k+1]
    u3 = U[4*k+2]
    u4 = U[4*k+3]

    x1 = -10 * math.log(1-u1)
    x2 = -8 * math.log(1-u2)
    x3 = -7 * math.log(1-u3)
    x4 = -5 * math.log(1-u4)

    x34 = max(x3,x4)
    x134 = min(x34,x1)
    x1234 = max(x134,x2)

    y_k = x1234

    if y_k >= 7:
        total+=1

p_hat = total/N
print(p_hat)

    





0.5709


## C1

In [3]:
from scipy import stats

def find_cycle(step, seed):
    
    # step is a function

    seen_at = {}
    curr = seed
    iteration = 0

    while curr not in seen_at:
        seen_at[curr] = iteration
        curr = step(curr)
        iteration+=1

    first_seen_iteration = seen_at[curr]
    cycle_len = iteration - first_seen_iteration
    return curr,iteration,cycle_len

def bin_counts(uniforms, bins=10):
    counts = [0] * bins
    for value in uniforms:
        index = min(
            bins-1, int(value*bins)
        ) # atmost bins-1 porjonto hobe
        counts[index] +=1
    return counts

def chi_square_statistic_manual(counts):
    """Step 1-4 without any library: chi2 = sum (O_i - E_i)^2 / E_i."""
    total = sum(counts)
    expected = total / len(counts)
    return sum(((observed - expected) ** 2) / expected for observed in counts)


def chi_square_uniform_test(uniforms, bins=10, alpha=0.05):
    """
    Full Chi-Square uniformity test on a list of values expected in [0, 1).

    Returns a dict with counts, the statistic, critical value, p-value,
    and the H0 decision -- this is exactly the row format the C1 table
    (Seed | Chi^2 | p-value | Decision) asks for.
    """
    counts = bin_counts(uniforms, bins)
    chi2 = chi_square_statistic_manual(counts)
    df = bins - 1
    critical_value = stats.chi2.ppf(1 - alpha, df=df)
    p_value = stats.chi2.sf(chi2, df=df)
    decision = "Reject H0" if p_value < alpha else "Do not reject H0"
    return {
        "counts": counts,
        "chi2": chi2,
        "df": df,
        "critical_value": critical_value,
        "p_value": p_value,
        "decision": decision,
    }

def middle_square_step(x):
    offset = 2
    total_digits = 4
    square = x*x
    str_sqr = str(square)
    while len(str_sqr)<8:
        str_sqr="0"+str_sqr
    # print(str_sqr)
    str_x = str_sqr[offset:offset+total_digits]
    # print(str_x)
    x = int(str_x)
    # print(x)
    return x

def middle_square_unifrom(seed,n):
    return [x / 10000 for x in middle_square(seed,n)]

def middle_square(seed, n):
    x = seed
    values = []

    for _ in range(n):
        # square = x*x
        # str_sqr = str(square)
        # while len(str_sqr)<8:
        #     str_sqr="0"+str_sqr
        # # print(str_sqr)
        # str_x = str_sqr[offset:offset+total_digits]
        # # print(str_x)
        # x = int(str_x)
        # # print(x)

        x = middle_square_step(x)
        values.append(x)
    return values

values = middle_square(5731,5)
print(values)

def find_ms_cycle(seed):
    return find_cycle(
        lambda x: middle_square_step(x), seed
    )

# print(find_ms_cycle(5731))

# samples = middle_square_unifrom(5731,20)
# print(samples)
# print(chi_square_uniform_test(samples))
problematic_seed = 1000
seeds = [5731, 6239, problematic_seed]


first_repeat, iteration, cycle_len = find_ms_cycle(seed=problematic_seed)
print(f"Problematic Seed: {problematic_seed}, First Repeated Value: {first_repeat}, Iteration: {iteration}, Cycle Length: {cycle_len}")
n = 1000
for seed in seeds:
    samples = middle_square_unifrom(seed, n)
    print(f" Seed: {seed}")
    print(chi_square_uniform_test(samples))




[8443, 2842, 769, 5913, 9635]
Problematic Seed: 1000, First Repeated Value: 0, Iteration: 2, Cycle Length: 1
 Seed: 5731
{'counts': [11, 10, 241, 3, 240, 6, 238, 5, 240, 6], 'chi2': 1302.52, 'df': 9, 'critical_value': np.float64(16.918977604620448), 'p_value': np.float64(8.835040552078124e-275), 'decision': 'Reject H0'}
 Seed: 6239
{'counts': [7, 10, 238, 14, 230, 10, 233, 9, 239, 10], 'chi2': 1215.8, 'df': 9, 'critical_value': np.float64(16.918977604620448), 'p_value': np.float64(4.7060467868980064e-256), 'decision': 'Reject H0'}
 Seed: 1000
{'counts': [1000, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'chi2': 9000.0, 'df': 9, 'critical_value': np.float64(16.918977604620448), 'p_value': np.float64(0.0), 'decision': 'Reject H0'}


## C2

In [4]:
def find_cycle(step, seed):
    
    # step is a function

    seen_at = {}
    curr = seed
    iteration = 0

    while curr not in seen_at:
        seen_at[curr] = iteration
        curr = step(curr)
        iteration+=1

    first_seen_iteration = seen_at[curr]
    cycle_len = iteration - first_seen_iteration
    return curr,iteration,cycle_len

def bin_counts(uniforms, bins=10):
    counts = [0] * bins
    for value in uniforms:
        index = min(
            bins-1, int(value*bins)
        ) # atmost bins-1 porjonto hobe
        counts[index] +=1
    return counts


def lcg_step(x, a, c, m):
    return (a * x + c) % m

def lcg(n, seed=1, a=5, c=0, m=65536):
    """Generate n successive integer states X1..Xn."""
    values = []
    x = seed
    for _ in range(n):
        x = lcg_step(x, a, c, m)
        values.append(x)
    return values

def lcg_uniforms(n, seed=1, a=5, c=0, m=65536):
    """Same sequence as lcg(), rescaled to floats in [0, 1)."""
    return [x / m for x in lcg(n, seed,a,c,m)]

def find_lcg_cycle(seed=1, a=5, c=0, m=65536):
    return find_cycle(
        step=lambda x: lcg_step(x,a,c,m),
        seed=seed
    )

from scipy import stats
import math

def ks_statistic(sample):
    """Manual D+, D-, D computation (matches the slide's table layout)."""
    n = len(sample)
    s = sorted(sample)
    d_plus = max(i / n - s[i - 1] for i in range(1, n + 1))
    d_minus = max(s[i - 1] - (i - 1) / n for i in range(1, n + 1))
    return d_plus, d_minus, max(d_plus, d_minus)


def ks_uniform_test(sample, alpha=0.05):
    """
    Full K-S uniformity test. Uses scipy's `ksone` distribution for the
    critical value / p-value -- exact for any N and alpha (no table needed).
    """
    d_plus, d_minus, d = ks_statistic(sample)
    n = len(sample)
    d_critical = 1.36/ math.sqrt(n)
    p_value = 1 - stats.ksone.cdf(d, n)
    # print(f"d_critical: {d_critical}")
    # print(f"p_value: {p_value}")
    # print(f"D value: {d}")
    decision = "Reject H0" if d > d_critical else "Do not reject H0"
    return {
        "D+": d_plus,
        "D-": d_minus,
        "D": d,
        "critical_value": d_critical,
        "p_value": p_value,
        "decision": decision,
    }

n = 100000
samples = lcg_uniforms(n)
print(ks_uniform_test(samples))

from math import gcd

def is_prime(n):
    if n < 2:
        return False
    for i in range(2, int(n**0.5) + 1):
        if n % i == 0:
            return False
    return True

def check_max_period_conditions(a=5, m=65536, seed=1,c=0):
    """
    Checks which theoretical max-period case applies and whether its
    conditions are satisfied. Returns (case_description, theoretical_max_period,
    conditions_satisfied, details_string). Falls back to "measure it yourself
    with find_lcg_cycle" when m doesn't match a known closed-form case
    (e.g. m composite and not a power of 2).
    """
    is_power_of_two = m > 0 and (m & (m - 1)) == 0

    if is_power_of_two and c != 0:
        cond1 = gcd(c, m) == 1
        cond2 = (a - 1) % 4 == 0
        return ("Case 1: m=2^b, mixed (c!=0)", m, cond1 and cond2,
                f"gcd(c,m)==1: {cond1}, (a-1)%4==0: {cond2}")

    if is_power_of_two and c == 0:
        cond1 = seed % 2 == 1
        cond2 = a % 8 in (3, 5)
        return ("Case 2: m=2^b, multiplicative (c=0)", m // 4, cond1 and cond2,
                f"seed is odd: {cond1}, a%8 in {{3,5}}: {cond2}")

    if c == 0 and is_prime(m):
        # a is a primitive root mod m  <=>  order of a divides (m-1) and equals it exactly.
        order, val = 1, a % m
        while val != 1 and order < m:
            val = (val * a) % m
            order += 1
        return ("Case 3: m prime, multiplicative (c=0)", m - 1, order == m - 1,
                f"multiplicative order of a mod m = {order}")

    return ("General case (no closed form here)", None, None,
            "use find_lcg_cycle(seed, a, c, m) to measure the period directly")

print(check_max_period_conditions())
first_repeat, iteration, cycle_len = find_lcg_cycle()
print(f"Seed: 1, First Repeated Value: {first_repeat}, Iteration: {iteration}, Cycle Length: {cycle_len}")

{'D+': 0.00046378417968750973, 'D-': 0.00022143066406254963, 'D': 0.00046378417968750973, 'critical_value': 0.0043006976178289955, 'p_value': np.float64(0.9575970086076443), 'decision': 'Do not reject H0'}
('Case 2: m=2^b, multiplicative (c=0)', 16384, True, 'seed is odd: True, a%8 in {3,5}: True')
Seed: 1, First Repeated Value: 1, Iteration: 16384, Cycle Length: 16384
